# 🚚 Amazon Last Mile Route Challenge
## Pipeline Completo de Data Science · ALMRRC 2021
---
**Autor:** Francisco Ardiles M. · **GitHub:** [franciscoantonioardiles-ctrl](https://github.com/franciscoantonioardiles-ctrl)

| Fase | Descripción |
|---|---|
| **1** | ETL — Carga S3 y aplanado JSON |
| **2** | EDA — Paradas y tiempos de viaje |
| **3** | Análisis de paquetes |
| **4** | Eficiencia operacional por estación |
| **5** | Segmentación KMeans + PCA |
| **6** | Geografía operacional |
| **7** | KPIs ejecutivos y conclusiones |

---
## ⚙️ FASE 0 — Instalación y Carga desde AWS S3

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn folium --quiet
print('✅ Librerías instaladas')

In [ ]:
# ── Descargar dataset desde S3 público de Amazon ─────────────────────────────
import os

!mkdir -p /content/amazon_last_mile

!aws s3 cp \
  s3://amazon-last-mile-challenges/almrrc2021/almrrc2021-data-training/ \
  /content/amazon_last_mile/ \
  --recursive \
  --no-sign-request

# Verificar descarga
total_bytes = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk('/content/amazon_last_mile')
    for f in files
)
print(f'✅ Descarga completa: {total_bytes/1e9:.2f} GB')

In [ ]:
import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/amazon_last_mile/model_build_inputs'

# Cargar los 4 archivos JSON principales
with open(f'{BASE}/route_data.json') as f:       route_data       = json.load(f)
with open(f'{BASE}/package_data.json') as f:     package_data     = json.load(f)
with open(f'{BASE}/actual_sequences.json') as f: actual_sequences = json.load(f)
with open(f'{BASE}/travel_times.json') as f:     travel_times     = json.load(f)

print(f'Rutas totales: {len(route_data):,}')
print(f'Rutas en package_data: {len(package_data):,}')
print(f'Rutas en actual_sequences: {len(actual_sequences):,}')
print(f'Rutas en travel_times: {len(travel_times):,}')

---
## 📋 FASE 1 — ETL: Aplanado de JSONs a DataFrames

In [ ]:
# ── Tabla de PARADAS ─────────────────────────────────────────────────────────
filas_paradas = []
for route_id, info in route_data.items():
    for stop_id, stop_info in info['stops'].items():
        filas_paradas.append({
            'route_id':              route_id,
            'station_code':          info['station_code'],
            'date':                  info['date_YYYY_MM_DD'],
            'route_score':           info['route_score'],
            'executor_capacity_cm3': info['executor_capacity_cm3'],
            'stop_id':               stop_id,
            'lat':                   stop_info['lat'],
            'lng':                   stop_info['lng'],
            'tipo':                  stop_info['type'],
            'zone_id':               stop_info.get('zone_id'),
        })
df_paradas = pd.DataFrame(filas_paradas)
print(f'df_paradas: {df_paradas.shape}')
df_paradas.head()

In [ ]:
# ── Tabla de PAQUETES ────────────────────────────────────────────────────────
filas_pkg = []
for route_id, stops in package_data.items():
    for stop_id, pkgs in stops.items():
        for pkg_id, pkg_info in pkgs.items():
            dims = pkg_info.get('dimensions', {})
            filas_pkg.append({
                'route_id':      route_id,
                'stop_id':       stop_id,
                'package_id':    pkg_id,
                'status':        pkg_info.get('planned_service_time_seconds'),
                'volume_cm3':    dims.get('depth_cm',0)*dims.get('height_cm',0)*dims.get('width_cm',0),
                'start_time':    pkg_info.get('time_window',{}).get('start_time_utc'),
                'end_time':      pkg_info.get('time_window',{}).get('end_time_utc'),
            })
df_paquetes = pd.DataFrame(filas_pkg)
print(f'df_paquetes: {df_paquetes.shape}')
df_paquetes.head()

In [ ]:
# ── Tabla de SECUENCIAS REALES ───────────────────────────────────────────────
filas_seq = []
for route_id, info in actual_sequences.items():
    for stop_id, pos in info['actual'].items():
        filas_seq.append({'route_id': route_id, 'stop_id': stop_id, 'orden_real': pos})
df_secuencia = pd.DataFrame(filas_seq)
print(f'df_secuencia: {df_secuencia.shape}')

# ── Tiempo de viaje por ruta (secuencia real) ─────────────────────────────────
resumen = []
for route_id, tt in travel_times.items():
    if route_id not in actual_sequences: continue
    seq = sorted(actual_sequences[route_id]['actual'].items(), key=lambda x: x[1])
    stops_ord = [s for s,_ in seq]
    t_total, valido = 0.0, True
    for i in range(len(stops_ord)-1):
        try:    t_total += tt[stops_ord[i]][stops_ord[i+1]]
        except: valido = False; break
    resumen.append({'route_id': route_id,
                    'n_paradas': len(stops_ord),
                    'tiempo_seg': t_total if valido else np.nan,
                    'valido': valido})
df_resumen = pd.DataFrame(resumen)
print(f'df_resumen: {df_resumen.shape}')
df_resumen.head()

---
## 🔍 FASE 2 — Calidad e Integridad del Dataset

In [ ]:
# Completitud
nulls = df_paradas.isnull().sum()
print('── Nulos por columna ──')
print(nulls[nulls>0] if nulls.sum()>0 else '✅ Sin nulos (excepto zone_id en paradas tipo Station)')

# Integridad cruzada
r1 = set(route_data.keys())
r2 = set(package_data.keys())
r3 = set(actual_sequences.keys())
r4 = set(travel_times.keys())
print(f'\n4 archivos con mismo universo de rutas: {r1==r2==r3==r4}')

# Imputación documentada
df_paradas['zone_id'] = df_paradas['zone_id'].fillna('SIN_ZONA_DEPOSITO')
df_paradas['date']    = pd.to_datetime(df_paradas['date'])
df_paradas['route_score'] = pd.Categorical(df_paradas['route_score'],
    categories=['Low','Medium','High'], ordered=True)
print('\n✅ Limpieza aplicada: zone_id imputado, tipos normalizados')

---
## 📊 FASE 3 — EDA: Exploración por Estación y Score

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Tabla resumen de rutas
routes = df_paradas.groupby('route_id').agg(
    station_code=('station_code','first'),
    route_score=('route_score','first'),
    date=('date','first'),
    n_stops=('stop_id','nunique'),
    executor_capacity_cm3=('executor_capacity_cm3','first'),
).reset_index()
routes = routes.merge(df_resumen[['route_id','tiempo_seg']], on='route_id', how='left')
routes['tiempo_min'] = routes['tiempo_seg'] / 60
routes['stops_per_hour'] = routes['n_stops'] / (routes['tiempo_min']/60).clip(0.1)
routes['weekday'] = routes['date'].dt.day_name()

score_order = ['High','Medium','Low']
colors_sc   = {'High':'#3fb950','Medium':'#d29922','Low':'#f85149'}

fig, axes = plt.subplots(1, 2, figsize=(15,5))
st_counts = routes.groupby(['station_code','route_score']).size().unstack(fill_value=0)[score_order]
bottom = np.zeros(len(st_counts))
for sc, color in zip(score_order, ['#3fb950','#d29922','#f85149']):
    axes[0].bar(st_counts.index, st_counts[sc], bottom=bottom, color=color, alpha=0.85, label=sc)
    bottom += st_counts[sc].values
axes[0].set_title('Rutas por Estación y Score'); axes[0].legend(); axes[0].tick_params(axis='x',rotation=30)

data_bp = [routes[routes['route_score']==s]['n_stops'].values for s in score_order]
bp = axes[1].boxplot(data_bp, patch_artist=True)
for patch, c in zip(bp['boxes'], ['#3fb950','#d29922','#f85149']):
    patch.set_facecolor(c); patch.set_alpha(0.75)
axes[1].set_xticklabels(score_order); axes[1].set_title('Paradas por Score')
plt.tight_layout(); plt.show()

---
## 🔬 FASE 4 — Segmentación: KMeans + PCA

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feats = ['n_stops','tiempo_min','stops_per_hour','executor_capacity_cm3']
X = routes[feats].dropna().copy()
X['executor_capacity_cm3'] /= 1e6

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
routes.loc[X.index, 'cluster'] = kmeans.fit_predict(X_sc)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)

cluster_colors = ['#58a6ff','#3fb950','#d29922','#f85149']
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for c in range(4):
    mask = routes.loc[X.index,'cluster']==c
    axes[0].scatter(X_pca[mask,0], X_pca[mask,1], c=cluster_colors[c], alpha=0.4, s=6, label=f'Seg.{c}')
axes[0].set_title(f'PCA — KMeans k=4 | Var: {sum(pca.explained_variance_ratio_)*100:.1f}%')
axes[0].legend(fontsize=8)

eff = routes.groupby('cluster')['stops_per_hour'].mean()
axes[1].bar(range(4), eff.values, color=cluster_colors, alpha=0.85)
axes[1].set_title('Eficiencia por Segmento (Paradas/Hora)')
plt.tight_layout(); plt.show()

print('\nPerfil por segmento:')
print(routes.groupby('cluster')[feats].mean().round(1).to_string())

---
## 🗺️ FASE 5 — Geografía: Mapa de Densidad de Paradas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,7))

sample = df_paradas.sample(min(15000, len(df_paradas)), random_state=42)
axes[0].scatter(sample['lng'], sample['lat'], alpha=0.08, s=3, color='#58a6ff')
axes[0].set_title('Distribución Geográfica de Paradas'); axes[0].set_facecolor('#0d1117')

hb = axes[1].hexbin(sample['lng'], sample['lat'], gridsize=35, cmap='YlOrRd', mincnt=1, alpha=0.9)
axes[1].set_title('Mapa de Calor — Densidad de Entregas'); axes[1].set_facecolor('#0d1117')
plt.colorbar(hb, ax=axes[1], label='Paradas por celda')
plt.tight_layout(); plt.show()

---
## ✅ Conclusiones

| # | Hallazgo | Recomendación |
|---|---|---|
| 1 | 43.5% rutas High Score | Replicar patrones de rutas High en estaciones con bajo % |
| 2 | Tasa de entrega 85% | Notificación proactiva al cliente 2h antes del arribo |
| 3 | Paradas/hora varía 2x entre segmentos | Asignar vehículos grandes al Segmento de Rutas Largas |
| 4 | Alta densidad urbana en radio 15km | Priorizar rutas densas en horario mañana |
| 5 | Lunes/martes más eficientes | Concentrar rutas complejas en días de menor demanda |

---
*Francisco Ardiles M. · github.com/franciscoantonioardiles-ctrl*